# FASE 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS DE MACHINE LEARNING (LOCAL)

## 📌 Contexto de Negocio
Para Telco, retener a un cliente existente es 5 veces más económico que adquirir uno nuevo a través de campañas de marketing. En las fases previas (EDA y Pipeline), identificamos los factores clave que impulsan la fuga de clientes (*Churn*) y transformamos esos datos crudos en un vector de características optimizado. En esta fase, utilizaremos esos datos para entrenar un algoritmo predictivo que funcione como un "radar de riesgo", permitiendo al equipo de Marketing actuar de forma proactiva y dirigida hacia los clientes con alta probabilidad de abandono.

## 🎯 Objetivo de este Notebook
El objetivo principal de este módulo es entrenar, tunear y evaluar un modelo de **Regresión Logística Binaria** utilizando la librería nativa `pyspark.ml`. Buscamos construir un modelo altamente preciso en nuestro entorno local (Victus) que clasifique correctamente si un cliente se quedará (*Churn = 0*) o se irá (*Churn = 1*), midiendo su éxito mediante métricas estándar de la industria.

## 📊 Estructura del Experimento de Machine Learning
El flujo de trabajo dentro de este laboratorio se divide en las siguientes etapas:
1. **Inicialización del Motor:** Activación de la sesión local de PySpark asignando 4GB de memoria RAM.
2. **Pipeline Integrado:** Ingesta del dataset final, imputación de nulos con la mediana calculada y vectorización de características (*Features* y *Label*).
3. **División Estratégica (Train/Test Split):** Separación aleatoria de los datos en un **70% para entrenamiento** (ajuste de pesos matemáticos) y un **30% para validación** (examen de datos nunca antes vistos).
4. **Entrenamiento Binario:** Ajuste del modelo de Regresión Logística forzando una familia binomial para asegurar una predicción probabilística de dos clases.
5. **Evaluación de Rendimiento:** Análisis de la capacidad de discriminación del modelo mediante el cálculo del **Área Bajo la Curva ROC (ROC AUC)** y la inspección de las columnas de probabilidad.

In [1]:
# ==============================================================================
# 1. INICIALIZACIÓN Y CONFIGURACIÓN DEL ENTORNO DE ML
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Iniciar o recuperar la sesión local de Spark
spark = SparkSession.builder \
    .appName("TelcoChurn_MachineLearning") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Sesión de Spark lista para Machine Learning. Versión: {spark.version}")

Sesión de Spark lista para Machine Learning. Versión: 3.5.0


In [3]:
# ==============================================================================
# 2. PIPELINE INTEGRADO Y DIVISIÓN DE DATOS (TRAIN/TEST SPLIT)
# ==============================================================================
from pyspark.sql.functions import col, when, count
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import ClassificationModel, LogisticRegression

# --- PASO 1: Ingesta Local ---s
csv_path = "../data/sample/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(csv_path)

# --- PASO 2: Limpieza con la Mediana ---
df_cleaned = df.withColumn("TotalCharges", col("TotalCharges").cast("double"))
mediana_total_charges = df_cleaned.approxQuantile("TotalCharges", [0.5], 0.001)[0]
df_cleaned = df_cleaned.na.fill({"TotalCharges": mediana_total_charges})

# --- PASO 3: Indexación de Textos ---
columnas_categoricas = [item[0] for item in df_cleaned.dtypes if item[1] == 'string' and item[0] != 'customerID']
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in columnas_categoricas]
df_indexed = df_cleaned
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

# --- PASO 4: Ensamblado Vectorial ---
columnas_features = [
    "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
    "gender_index", "Partner_index", "Dependents_index", "PhoneService_index",
    "MultipleLines_index", "InternetService_index", "OnlineSecurity_index",
    "OnlineBackup_index", "DeviceProtection_index", "TechSupport_index",
    "StreamingTV_index", "StreamingMovies_index", "Contract_index",
    "PaperlessBilling_index", "PaymentMethod_index"
]
assembler = VectorAssembler(inputCols=columnas_features, outputCol="features", handleInvalid="skip")
df_final = assembler.transform(df_indexed)
df_vectorizado = df_final.select("features", col("Churn_index").alias("label"))

# --- PASO 5: División de Datos (70% Train, 30% Test) ---
# Aseguramos que la etiqueta sea estrictamente un entero binario (0 o 1)
df_vectorizado = df_final.select("features", col("Churn_index").cast("int").alias("label"))

train_data, test_data = df_vectorizado.randomSplit([0.7, 0.3], seed=42)

# --- PASO 6: Configurar y Entrenar el Modelo Binario ---
print("\nEntrenando el modelo de Regresión Logística Binaria en tu Victus...")

# Agregamos 'family="binomial"' para forzar a Spark a entender que es SÍ o NO (Clasificación Binaria)
lr = LogisticRegression(
    featuresCol="features", 
    labelCol="label", 
    maxIter=10,
    family="binomial"
)

lr_model = lr.fit(train_data)
print("¡Modelo Binario entrenado con éxito en local!")


Entrenando el modelo de Regresión Logística Binaria en tu Victus...
¡Modelo Binario entrenado con éxito en local!


In [4]:
# ==============================================================================
# 3. EVALUACIÓN DE RENDIMIENTO (MÉTRICAS DEL MODELO)
# ==============================================================================
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Paso A: Hacer las predicciones usando el grupo de prueba (Test)
predictions = lr_model.transform(test_data)

# Mostrar cómo se ven las columnas de predicción que genera Spark
print("Muestra de predicciones (Etiqueta Real vs Predicción del Modelo):")
predictions.select("label", "rawPrediction", "probability", "prediction").show(5, truncate=False)

# Paso B: Configurar el evaluador para clasificación binaria usando la Curva ROC
evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction", 
    labelCol="label", 
    metricName="areaUnderROC"
)

# Paso C: Calcular el área bajo la curva (ROC AUC)
roc_auc = evaluator.evaluate(predictions)
print(f"-> El Área Bajo la Curva ROC (ROC AUC) del modelo es: {round(roc_auc, 4)}")

Muestra de predicciones (Etiqueta Real vs Predicción del Modelo):
+-----+----------------------------------------+----------------------------------------+----------+
|label|rawPrediction                           |probability                             |prediction|
+-----+----------------------------------------+----------------------------------------+----------+
|0    |[-1.1783214954723111,1.1783214954723111]|[0.23535413002120886,0.7646458699787911]|1.0       |
|0    |[-0.9017313356131261,0.9017313356131261]|[0.28869483735893875,0.7113051626410613]|1.0       |
|1    |[-1.112684587661691,1.112684587661691]  |[0.24737073727069545,0.7526292627293045]|1.0       |
|1    |[-1.112684587661691,1.112684587661691]  |[0.24737073727069545,0.7526292627293045]|1.0       |
|1    |[-1.1148412420514648,1.1148412420514648]|[0.24696943348660994,0.7530305665133901]|1.0       |
+-----+----------------------------------------+----------------------------------------+----------+
only showing top 5 rows

